# Calphy Executor Integration Demo

This notebook demonstrates the new executorlib support in phase_diagram_workflows.
It shows both the traditional usage and the new executor-based parallel execution.


In [ ]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase_diagram_workflows import calculator as calphy_calc
from executorlib import SingleNodeExecutor
from pylammpsmpi import LammpsLibrary, init_function

EXAMPLE_ROOT = PROJECT_ROOT / "notebooks" / "Aluminium_free_energy_executor"

In [ ]:
import os
from typing import Dict, Any, Optional
from ase.atoms import Atoms
from ase.build import bulk
from pyiron_lammps import get_potential_by_name
import pandas as pd

## 1. Structure and Potential Setup

In [ ]:
# Create Aluminum structure
Al_pure_structure = bulk('Al', cubic=True).repeat(3)
print(f"Created Al structure with {len(Al_pure_structure)} atoms")

In [ ]:
# Get potential for Al
potential_df = get_potential_by_name('2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1')
potential_df = potential_df.to_frame().transpose()
print("Potential loaded successfully:")
display(potential_df.head())

## 2. Traditional Usage (Backward Compatible)

This shows the original API that continues to work unchanged.

In [ ]:
# Parameters for free energy calculation
input_params_fe = {  
    "mode": "fe", 
    "temperature": 300,  
    "n_equilibration_steps": 2500,
    "n_switching_steps": 2500,
    "n_print_steps": 1000,
    "equilibration_control": "nose-hoover",
    "queue": {
        "cores": 1,  
        "scheduler": "local"  
    },
    'reference_phase': 'solid',
    'file_format': 'lammps-data',
}

In [ ]:
# Traditional usage - no executor, no metadata
print("Running traditional calphy calculation...")
calphy_results_traditional = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_fe,
    working_directory=str(EXAMPLE_ROOT / "Al_traditional"),
)
print("Traditional calculation completed successfully!")

In [ ]:
# Show traditional results
print("Calculation object:")
display(calphy_results_traditional[0])
print("\nResults DataFrame:")
display(calphy_results_traditional[1])

## 3. New Executor-Based Usage

This demonstrates the new executorlib integration with parallel execution.

In [ ]:
# Set up executor and LAMMPS library
print("Setting up executor...")
executor = SingleNodeExecutor(
    block_allocation=True,
    hostname_localhost=True,
    max_workers=1,
    init_function=init_function,
    resource_dict={
        "cores": 1,
        "cwd": str(EXAMPLE_ROOT),
    },
)

# Create LAMMPS library with executor
lmp = LammpsLibrary(cores=1, executor=executor)
print("Executor and LAMMPS library ready!")

In [ ]:
# New usage with executor and metadata
print("Running calphy calculation with executor...")
metadata_dict = {
    'project': 'calphy-executor-demo',
    'material': 'Aluminum',
    'temperature': 300,
    'method': 'executor-based',
    'version': '1.0'
}

calphy_results_executor = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_fe,
    working_directory=str(EXAMPLE_ROOT / "Al_executor"),
    lmp=lmp,  # Pass the LAMMPS library with executor
    metadata_dict=metadata_dict  # Pass metadata for caching
)
print("Executor-based calculation completed successfully!")

In [ ]:
# Show executor results
print("Calculation object:")
display(calphy_results_executor[0])
print("\nResults DataFrame:")
display(calphy_results_executor[1])
print("\nMetadata used:")
print(metadata_dict)

## 4. Comparison and Benefits

### Key Differences:
- **Traditional**: Runs sequentially, no parallel execution
- **Executor-based**: Runs with parallel execution via executorlib

### Benefits of Executor Integration:
- ✅ Parallel execution for faster calculations
- ✅ Resource management via executorlib
- ✅ Metadata support for result caching and retrieval
- ✅ Better resource utilization
- ✅ Full backward compatibility maintained

In [ ]:
# Clean up executor
if 'executor' in locals():
    executor.shutdown()
    print("Executor shutdown complete")

print("Demo completed successfully! 🎉")